# Indonesia Stock Universe — official registry plus market snapshot

> **Educational research universe—not a buy list.** This notebook separates an official KSEI security registry from a Yahoo Finance daily-price snapshot. Registration, price availability, size, or sector membership does not establish suitability, liquidity, or current tradability.

The table is point-in-time: KSEI fields and Yahoo prices have explicit retrieval dates. Yahoo's latest daily bar can be delayed, stale, or missing. Bulk Yahoo downloads remain under the Git-ignored `private/` directory because free access is not an open-data redistribution licence.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

# Objective: locate the same artifact from either the repository root or notebook directory.
def locate(relative_path, private=False):
    candidates = [Path(relative_path), Path('products/stocks') / relative_path]
    if private:
        candidates.extend([
            Path('../../private') / relative_path,
            Path('private') / relative_path,
        ])
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not locate {relative_path}; tried {candidates}')

master_path = locate('data/processed/ksei_registered_share_securities_2026-09-22.csv')
manifest_path = locate('data/manifests/ksei_stock_universe_2026-09-22.json')
price_path = locate('market_data/yahoo/idx_latest_prices_2026-09-22.csv', private=True)
price_manifest_path = locate('market_data/yahoo/idx_latest_prices_2026-09-22.json', private=True)
metadata_path = locate('market_data/yahoo/idx_company_metadata_2026-09-22.csv', private=True)
metadata_manifest_path = locate('market_data/yahoo/idx_company_metadata_2026-09-22.json', private=True)

registered = pd.read_csv(master_path)
ksei_manifest = json.loads(manifest_path.read_text())
prices = pd.read_csv(price_path)
price_manifest = json.loads(price_manifest_path.read_text())
company_metadata = pd.read_csv(metadata_path)
metadata_manifest = json.loads(metadata_manifest_path.read_text())
len(registered), len(prices), len(company_metadata)

(984, 980, 919)

## Coverage and scope

KSEI registered share securities are broader than active ordinary shares listed on IDX. Because KSEI's detail endpoint was throttled during this refresh, the **current market-data layer** means four-letter KSEI candidates for which Yahoo returned a recent daily price. That is a transparent operational definition, not an official listed-company count. A missing Yahoo row is retained as missing and never converted to a zero price.

In [2]:
summary = pd.Series({
    'KSEI registered share securities': len(registered),
    'standard four-letter ticker candidates': int(registered['ticker'].astype(str).str.fullmatch(r'[A-Z]{4}').sum()),
    'Yahoo price candidates requested': price_manifest['requested_ticker_count'],
    'Yahoo daily prices available': int((prices['price_status'] == 'available').sum()),
    'Yahoo daily prices missing': int((prices['price_status'] != 'available').sum()),
    'Yahoo company metadata available': int((company_metadata['metadata_status'] == 'available').sum()),
    'Yahoo company metadata errors after retry': int((company_metadata['metadata_status'] != 'available').sum()),
    'KSEI snapshot date': ksei_manifest['snapshot_date'],
    'Yahoo retrieval time UTC': price_manifest['retrieved_at_utc'],
}, name='value').to_frame()
summary

,value
KSEI registered share securities,984
standard four-letter ticker candidates,980
Yahoo price candidates requested,980
Yahoo daily prices available,919
Yahoo daily prices missing,61
Yahoo company metadata available,919
Yahoo company metadata errors after retry,0
KSEI snapshot date,2026-09-22
Yahoo retrieval time UTC,2026-09-22T02:30:16.136258+00:00


## Build the current research table

Yahoo supplies provider-reported `sharesOutstanding`, `floatShares`, and `marketCap`. The notebook also calculates `market_cap_recomputed_idr = sharesOutstanding × latest unadjusted close` as a consistency check. These fields can have different effective timestamps and are not official IDX or audited values.

In [3]:
current = registered.loc[
    registered['ticker'].astype(str).str.fullmatch(r'[A-Z]{4}')
].merge(
    prices.loc[prices['price_status'].eq('available')],
    on='ticker',
    how='inner',
    validate='one_to_one',
    suffixes=('_ksei', '_yahoo'),
)
current = current.merge(
    company_metadata.drop_duplicates('ticker'),
    on='ticker',
    how='left',
    validate='one_to_one',
    suffixes=('', '_metadata'),
)
current['market_cap_recomputed_idr'] = current['sharesOutstanding'] * current['close_idr']
current['market_cap_trillion_idr'] = current['marketCap'] / 1e12
current['market_cap_recomputed_trillion_idr'] = current['market_cap_recomputed_idr'] / 1e12
current['market_cap_difference_pct'] = 100 * (
    current['market_cap_recomputed_idr'] / current['marketCap'] - 1
)
current['price_age_days_at_snapshot'] = (
    pd.Timestamp(ksei_manifest['snapshot_date']) - pd.to_datetime(current['price_date'])
).dt.days.astype('Int64')

sector_column = 'sector'
current[sector_column] = current[sector_column].fillna('Unclassified by Yahoo')
current['industry'] = current['industry'].fillna('Unclassified by Yahoo')
current.shape

(919, 45)

## Sector and industry catalogue

These are **Yahoo sector and industry classifications**, not IDX-IC. They are useful for navigation and comparison but must not be presented as the exchange's official taxonomy. The notebook will adopt IDX-IC only after a reproducible official IDX extract becomes available.

In [4]:
sector_summary = (
    current.groupby(sector_column, dropna=False)
    .agg(
        securities=('ticker', 'size'),
        with_price=('close_idr', 'count'),
        industries=('industry', 'nunique'),
        with_issued_shares=('sharesOutstanding', 'count'),
        market_cap_coverage=('marketCap', 'count'),
        reported_market_cap_idr=('marketCap', 'sum'),
    )
    .sort_values(['reported_market_cap_idr', 'securities'], ascending=False)
)
sector_summary['reported_market_cap_trillion_idr'] = sector_summary['reported_market_cap_idr'] / 1e12
sector_summary

,securities,with_price,industries,with_issued_shares,market_cap_coverage,reported_market_cap_idr,reported_market_cap_trillion_idr
sector,,,,,,,
Financial Services,103,103,10,103,103,"2,789,646,428,372,992.00","2,789.65"
Basic Materials,92,92,12,92,92,"1,756,406,061,785,088.00","1,756.41"
Energy,57,57,6,57,57,"1,315,759,593,705,472.00","1,315.76"
Consumer Defensive,124,124,11,124,124,"1,039,391,461,560,064.00","1,039.39"
Real Estate,89,89,3,89,89,"1,011,990,108,424,192.00","1,011.99"
Communication Services,48,48,6,48,48,"962,120,755,855,360.00",962.12
Industrials,168,168,22,168,168,"633,400,079,869,952.00",633.40
Utilities,13,13,4,13,13,"621,977,879,339,008.00",621.98
Consumer Cyclical,143,143,21,143,143,"444,465,783,926,272.00",444.47


## Largest securities by estimated market capitalization

In [5]:
display_columns = [
    'ticker', 'registry_name', sector_column, 'industry', 'price_date', 'close_idr',
    'sharesOutstanding', 'floatShares', 'market_cap_trillion_idr',
    'market_cap_recomputed_trillion_idr', 'market_cap_difference_pct', 'volume_shares'
]
largest_by_market_cap = current.dropna(subset=['marketCap']).sort_values(
    ['marketCap', 'ticker'], ascending=[False, True]
)
largest_by_market_cap[display_columns].head(50)

,ticker,registry_name,sector,industry,price_date,close_idr,sharesOutstanding,floatShares,market_cap_trillion_idr,market_cap_recomputed_trillion_idr,market_cap_difference_pct,volume_shares
95,BBCA,BANK CENTRAL ASIA Tbk,Financial Services,Banks - Regional,2026-09-22,"6,250.00","122,841,751,300.00","48,199,417,958.00",767.76,767.76,-0.00,"15,300,300.00"
101,BBRI,BANK RAKYAT INDONESIA (PERSERO) Tbk,Financial Services,Banks - Regional,2026-09-22,"3,260.00","149,878,079,535.00","69,249,932,878.00",488.22,488.60,0.08,"37,461,100.00"
244,DCII,DCI INDONESIA Tbk,Real Estate,Real Estate Services,2026-09-22,"201,775.00","2,383,745,900.00","533,077,096.00",480.98,480.98,0.00,200.00
189,BYAN,BAYAN RESOURCES Tbk,Energy,Thermal Coal,2026-09-22,"13,400.00","33,333,335,000.00","7,092,667,021.00",440.00,446.67,1.52,"1,478,700.00"
164,BREN,BARITO RENEWABLES ENERGY Tbk,Utilities,Utilities - Renewable,2026-09-22,"3,110.00","133,779,520,000.00","16,832,139,206.00",413.38,416.05,0.65,"648,400.00"
146,BMRI,BANK MANDIRI ( PERSERO ) Tbk,Financial Services,Banks - Regional,2026-09-22,"4,210.00","92,487,999,999.00","37,719,733,333.00",391.42,389.37,-0.52,"24,686,000.00"
36,AMMN,AMMAN MINERAL INTERNASIONAL Tbk,Basic Materials,Other Precious Metals & Mining,2026-09-22,"4,810.00","72,412,413,856.00","14,738,822,716.00",349.03,348.30,-0.21,"19,985,300.00"
837,TLKM,TELKOM INDONESIA (PERSERO) Tbk,Communication Services,Telecom Services,2026-09-22,"2,520.00","98,580,940,799.00","46,453,310,924.00",245.47,248.42,1.20,"18,251,600.00"
573,MORA,EKAMAS MORA REPUBLIK Tbk,Communication Services,Telecom Services,2026-09-22,"4,890.00","47,774,192,736.00","9,020,723,072.00",236.00,233.62,-1.01,"6,200.00"
62,ASII,ASTRA INTERNATIONAL Tbk,Industrials,Conglomerates,2026-09-22,"4,870.00","39,920,061,040.00","17,653,848,594.00",193.21,194.41,0.62,"4,410,500.00"


## Highest nominal share prices

A high rupiah price per share does **not** make a company larger or more expensive by valuation. Share price depends on the number of issued shares and past splits. Use the market-cap table for size and valuation ratios for relative valuation.

In [6]:
highest_share_prices = current.dropna(subset=['close_idr']).sort_values(
    ['close_idr', 'ticker'], ascending=[False, True]
)
highest_share_prices[[
    'ticker', 'registry_name', sector_column, 'price_date', 'close_idr',
    'volume_shares', 'sharesOutstanding', 'floatShares', 'market_cap_trillion_idr'
]].head(50)

,ticker,registry_name,sector,price_date,close_idr,volume_shares,sharesOutstanding,floatShares,market_cap_trillion_idr
244,DCII,DCI INDONESIA Tbk,Real Estate,2026-09-22,"201,775.00",200.00,"2,383,745,900.00","533,077,096.00",480.98
807,SUPR,SOLUSI TUNAS PRATAMA Tbk,Real Estate,2026-09-21,"43,850.00",0.00,"1,137,579,698.00","978,319.00",49.88
9,ADES,AKASHA WIRA INTERNATIONAL Tbk,Consumer Defensive,2026-09-21,"31,975.00","5,000.00","589,896,800.00","50,990,679.00",18.86
431,ITMG,INDO TAMBANGRAYA MEGAH Tbk,Energy,2026-09-22,"26,150.00","80,300.00","1,114,899,100.00","365,258,447.00",29.10
873,UNTR,UNITED TRACTORS Tbk,Basic Materials,2026-09-22,"24,950.00","522,500.00","3,493,093,536.00","1,272,778,492.00",86.54
563,MKPI,METROPOLITAN KENTJANA Tbk,Real Estate,2026-09-22,"21,425.00","41,000.00","948,194,000.00","156,831,288.00",20.27
777,SMMA,SINAR MAS MULTIARTHA Tbk,Financial Services,2026-09-21,"20,025.00","1,200.00","142,474,368.00","50,495,581.00",127.51
327,GGRM,GUDANG GARAM Tbk,Consumer Defensive,2026-09-22,"17,850.00","50,700.00","1,924,088,000.00","457,528,886.00",34.59
674,POLU,GOLDEN FLOWER Tbk,Consumer Cyclical,2026-09-22,"16,800.00","1,900.00","750,000,000.00","85,897,500.00",12.49
550,MGLV,NEXAI DIGITAL INFRASTRUKTUR Tbk,Consumer Cyclical,2026-09-22,"16,375.00","1,150,000.00","1,904,883,411.00","664,880,506.00",30.95


## Browse every current security, organized by sector and size

In [7]:
all_current_stocks = current.sort_values(
    [sector_column, 'marketCap', 'close_idr', 'ticker'],
    ascending=[True, False, False, True],
    na_position='last',
)
all_current_stocks[[
    'ticker', 'registry_name', sector_column, 'price_status', 'price_date',
    'close_idr', 'sharesOutstanding', 'floatShares', 'market_cap_trillion_idr',
    'volume_shares', 'currency', 'exchange', 'detail_url'
]]

,ticker,registry_name,sector,price_status,price_date,close_idr,sharesOutstanding,floatShares,market_cap_trillion_idr,volume_shares,currency,exchange,detail_url
36,AMMN,AMMAN MINERAL INTERNASIONAL Tbk,Basic Materials,available,2026-09-22,"4,810.00","72,412,413,856.00","14,738,822,716.00",349.03,"19,985,300.00",IDR,JKT,https://web.ksei.co.id/services/registered-sec...
849,TPIA,CHANDRA ASRI PACIFIC Tbk,Basic Materials,available,2026-09-22,"1,875.00","86,464,496,192.00","25,432,666,910.00",160.39,"16,684,300.00",IDR,JKT,https://web.ksei.co.id/services/registered-sec...
168,BRPT,BARITO PACIFIC Tbk,Basic Materials,available,2026-09-22,"1,625.00","93,711,740,929.00","26,461,384,286.00",151.34,"4,043,400.00",IDR,JKT,https://web.ksei.co.id/services/registered-sec...
286,EMAS,MERDEKA GOLD RESOURCES Tbk,Basic Materials,available,2026-09-22,"7,650.00","14,731,366,060.00","2,662,252,474.00",112.33,"522,500.00",IDR,JKT,https://web.ksei.co.id/services/registered-sec...
166,BRMS,BUMI RESOURCES MINERALS Tbk,Basic Materials,available,2026-09-22,695.00,"25,570,150,644.00","24,778,178,889.00",97.83,"20,720,900.00",IDR,JKT,https://web.ksei.co.id/services/registered-sec...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
321,FUTR,FUTURA ENERGI GLOBAL Tbk,Utilities,available,2026-09-22,264.00,"6,635,551,959.00","3,061,046,474.00",1.74,"6,971,600.00",IDR,JKT,https://web.ksei.co.id/services/registered-sec...
365,HGII,HERO GLOBAL INVESTMENT Tbk,Utilities,available,2026-09-22,141.00,"6,500,000,000.00","1,061,970,000.00",0.93,"139,100.00",IDR,JKT,https://web.ksei.co.id/services/registered-sec...
783,SOFA,SOLUSI ENVIRONMENT ASIA Tbk,Utilities,available,2026-09-22,348.00,"1,653,574,499.00","480,082,284.00",0.57,"839,100.00",IDR,JKT,https://web.ksei.co.id/services/registered-sec...
576,MPOW,MEGAPOWER MAKMUR Tbk,Utilities,available,2026-09-22,113.00,"816,997,053.00","550,843,923.00",0.09,"111,500.00",IDR,JKT,https://web.ksei.co.id/services/registered-sec...


## Coverage exceptions requiring review

In [8]:
coverage_exceptions = current.loc[
    current['sharesOutstanding'].isna()
    | current['marketCap'].isna()
    | current[sector_column].eq('Unclassified by Yahoo')
].copy()
coverage_exceptions[[
    'ticker', 'registry_name', sector_column, 'industry', 'metadata_status',
    'price_date', 'close_idr', 'sharesOutstanding', 'marketCap', 'detail_url'
]].sort_values(['metadata_status', 'ticker'])

,ticker,registry_name,sector,industry,metadata_status,price_date,close_idr,sharesOutstanding,marketCap,detail_url
159,BOSS,BORNEO OLAH SARANA SUKSES Tbk,Unclassified by Yahoo,Unclassified by Yahoo,available,2026-09-21,50.00,"1,400,000,000.00","70,000,001,024.00",https://web.ksei.co.id/services/registered-sec...
245,DEAL,DEWATA FREIGHTINTERNATIONAL Tbk,Unclassified by Yahoo,Unclassified by Yahoo,available,2026-09-21,6.00,"1,146,170,959.00","6,877,026,304.00",https://web.ksei.co.id/services/registered-sec...
302,ETWA,ETERINDO WAHANATAMA Tbk,Unclassified by Yahoo,Unclassified by Yahoo,available,2026-09-21,70.00,"4,668,671,400.00","326,806,994,944.00",https://web.ksei.co.id/services/registered-sec...
452,KAYU,DARMI BERSAUDARA Tbk,Unclassified by Yahoo,Unclassified by Yahoo,available,2026-09-21,18.00,"665,000,000.00","11,969,999,872.00",https://web.ksei.co.id/services/registered-sec...
493,LAPD,LEYAND INTERNATIONAL Tbk,Unclassified by Yahoo,Unclassified by Yahoo,available,2026-09-22,79.00,"3,966,350,139.00","309,375,303,680.00",https://web.ksei.co.id/services/registered-sec...


## Interpretation rules

- The table is a research inventory, not a recommendation or order list.
- KSEI registration and Yahoo price availability do not guarantee current IDX listing, normal liquidity, or immediate purchasability.
- Yahoo's last daily bar is not an official real-time IDX quote; verify the price before any decision.
- Yahoo market capitalization and share counts are provider metadata; verify important figures against dated issuer and IDX disclosures.
- Nominal price should never be used alone to decide whether a stock is “cheap.” Compare business quality, balance sheet, cash flow, per-share fundamentals, valuation, governance, liquidity, and risk.
- A historical backtest needs point-in-time membership, delistings, suspensions, corporate actions, costs, and publication dates; this current snapshot must not be projected backward.

## Sources and reproducible refresh

- KSEI registered-share master: `scripts/fetch_ksei_stock_universe.py`
- Yahoo daily snapshot: `scripts/fetch_yahoo_stock_snapshot.py`
- Yahoo sector, industry, market-cap, and share metadata: `scripts/fetch_yahoo_company_snapshot.py`
- Official IDX monthly stock-price table was attempted for IDX-IC classification, but its table endpoint remained stuck at “Loading Data” and the Excel action did not produce a file. KSEI detail enrichment was also throttled. Yahoo classifications are therefore retained and labelled honestly rather than misrepresented as IDX-IC.
- Yahoo's actual data rights are separate from the Apache-licensed `yfinance` client. The bulk Yahoo cache remains under `private/`.

Refresh prices with:

```bash
PYTHONNOUSERSITE=1 uv run --isolated --with 'numpy<2' --with yfinance \
  python scripts/fetch_yahoo_stock_snapshot.py --as-of YYYY-MM-DD

PYTHONNOUSERSITE=1 uv run --isolated --with 'numpy<2' --with yfinance \
  python scripts/fetch_yahoo_company_snapshot.py --as-of YYYY-MM-DD --workers 6
```
